# 05.1 - Caracterização do Dataset Final

Análise descritiva e estatísticas de heterogeneidade da base canônica final (`experiments.parquet`).

Objetivos:
1. Resumo geral (volume, cobertura temporal, dimensionalidade)
2. Distribuições dimensionais (task types, modelos, estratégias)
3. Heterogeneidade cruzada: estratégias vs. Imbalance Ratio
4. Heterogeneidade cruzada: estratégias vs. Família de modelo
5. Heterogeneidade cruzada: estratégias vs. Tipo de tarefa
6. Cobertura de dados estruturados e métricas

## Setup

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

# ── Paths
NB_DIR        = Path().resolve()
PROJECT_DIR   = NB_DIR.parent if NB_DIR.name == "notebooks" else NB_DIR
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Carregar com fastparquet
parquet_file = PROCESSED_DIR / "experiments.parquet"
exp = pd.read_parquet(str(parquet_file), engine='fastparquet')

print(f"✓ Dataset carregado: {exp.shape}")
print(f"  Colunas: {exp.shape[1]}")
print(f"  Linhas: {exp.shape[0]:,}")

✓ Dataset carregado: (1309, 28)
  Colunas: 28
  Linhas: 1,309


## 1. Resumo Descritivo Geral

In [23]:
print(f"{'='*60}")
print("  DATASET FINAL: Caracterização")
print(f"{'='*60}")
print(f"Experimentos:              {len(exp):,}")
print(f"Papers únicos:             {exp['paper_id'].nunique():,}")
print(f"Período:                   {exp['year'].min():.0f} – {exp['year'].max():.0f}")
print(f"Datasets únicos:           {exp['dataset_canonical'].nunique():,}")
print(f"Modelos únicos (family):   {exp['model_family'].nunique():,}")
print(f"Estratégias únicas:        {exp['balancing_strategy'].nunique():,}")
print()

  DATASET FINAL: Caracterização
Experimentos:              1,309
Papers únicos:             199
Período:                   2007 – 2026
Datasets únicos:           313
Modelos únicos (family):   11
Estratégias únicas:        11



## 2. Distribuições Dimensionais

In [24]:
# Task types
print("Distribuição por tipo de tarefa:")
for task, cnt in exp['task_type'].value_counts().items():
    print(f"  {task:<20} {cnt:>6}  ({cnt/len(exp):.1%})")
print()

Distribuição por tipo de tarefa:
  tabular                 804  (61.4%)
  other                   363  (27.7%)
  vision                  118  (9.0%)
  text                     20  (1.5%)
  time_series               4  (0.3%)



In [25]:
# Modelos
print("Top 10 famílias de modelo:")
for model, cnt in exp['model_family'].value_counts().head(10).items():
    print(f"  {model:<20} {cnt:>6}  ({cnt/len(exp):.1%})")
print()

Top 10 famílias de modelo:
  cnn                     297  (22.7%)
  ensemble                228  (17.4%)
  other                   207  (15.8%)
  kernel                  183  (14.0%)
  mlp                     161  (12.3%)
  tree                    109  (8.3%)
  linear                   50  (3.8%)
  gbm                      37  (2.8%)
  transformer              15  (1.1%)
  rnn                      14  (1.1%)



In [26]:
# Estratégias
print("Distribuição de estratégias:")
for strat, cnt in exp['balancing_strategy'].value_counts().items():
    print(f"  {strat:<25} {cnt:>6}  ({cnt/len(exp):.1%})")
print()

Distribuição de estratégias:
  none                         382  (29.2%)
  oversampling                 307  (23.5%)
  other                        285  (21.8%)
  cost_sensitive               112  (8.6%)
  undersampling                 74  (5.7%)
  ensemble_based                61  (4.7%)
  threshold_moving              33  (2.5%)
  hybrid                        20  (1.5%)
  generative                    18  (1.4%)
  data_augmentation             15  (1.1%)
  two_stage                      2  (0.2%)



## 3. Heterogeneidade: Estratégias × Imbalance Ratio

In [27]:
# Criar bins de IR
exp['ir_bin'] = pd.cut(
    exp['dataset_imbalance_ratio'],
    bins=[0, 1.5, 5, 10, 50, np.inf],
    labels=['1.0-1.5 (Low)', '1.5-5', '5-10', '10-50', '>50 (High)'],
    include_lowest=True
)

print("Heterogeneidade: Estratégias por faixa de Imbalance Ratio (contagem)")
crosstab_ir = pd.crosstab(
    exp['ir_bin'], 
    exp['balancing_strategy'],
    margins=True
)
print(crosstab_ir)
print()

Heterogeneidade: Estratégias por faixa de Imbalance Ratio (contagem)
balancing_strategy  cost_sensitive  data_augmentation  ensemble_based  \
ir_bin                                                                  
1.0-1.5 (Low)                    2                  0               1   
1.5-5                           13                  0              12   
5-10                            25                  2              16   
10-50                            8                  1              22   
>50 (High)                      47                 11               6   
All                             95                 14              57   

balancing_strategy  generative  hybrid  none  other  oversampling  \
ir_bin                                                              
1.0-1.5 (Low)                0       1    13     13            22   
1.5-5                        0       1    89     45            89   
5-10                         2       3    55     78            45   
1

In [28]:
# Percentuais (por linha = para cada faixa de IR, qual % de cada estratégia?)
print("Heterogeneidade: Estratégias por faixa de IR (% por linha)")
pct_ir = pd.crosstab(
    exp['ir_bin'], 
    exp['balancing_strategy'],
    normalize='index'
) * 100
print((pct_ir.round(1).astype(str) + '%').to_string())
print()

Heterogeneidade: Estratégias por faixa de IR (% por linha)
balancing_strategy cost_sensitive data_augmentation ensemble_based generative hybrid   none  other oversampling threshold_moving two_stage undersampling
ir_bin                                                                                                                                                  
1.0-1.5 (Low)                3.4%              0.0%           1.7%       0.0%   1.7%  22.0%  22.0%        37.3%             0.0%      0.0%         11.9%
1.5-5                        5.0%              0.0%           4.6%       0.0%   0.4%  34.0%  17.2%        34.0%             0.4%      0.0%          4.6%
5-10                        10.4%              0.8%           6.7%       0.8%   1.2%  22.9%  32.5%        18.8%             1.2%      0.0%          4.6%
10-50                        4.1%              0.5%          11.3%       0.5%   2.6%  26.7%  24.1%        23.6%             0.0%      0.0%          6.7%
>50 (High)             

## 4. Heterogeneidade: Estratégias × Família de Modelo

In [29]:
# Top modelos (para legibilidade)
top_models = exp['model_family'].value_counts().head(5).index
exp_top_models = exp[exp['model_family'].isin(top_models)]

print("Heterogeneidade: Estratégias por família de modelo - Top 5 (contagem)")
crosstab_model = pd.crosstab(
    exp_top_models['model_family'],
    exp_top_models['balancing_strategy'],
    margins=True
)
print(crosstab_model)
print()

Heterogeneidade: Estratégias por família de modelo - Top 5 (contagem)
balancing_strategy  cost_sensitive  data_augmentation  ensemble_based  \
model_family                                                            
cnn                             76                 11               3   
ensemble                         0                  0              24   
kernel                           8                  0               9   
mlp                              9                  0               0   
other                           12                  3               2   
All                            105                 14              38   

balancing_strategy  generative  hybrid  none  other  oversampling  \
model_family                                                        
cnn                          3       0    89     70            12   
ensemble                     2       4    68     43            72   
kernel                       1       3    56     38            58   


In [30]:
# Percentuais
print("Heterogeneidade: Estratégias por modelo - Top 5 (% por linha)")
pct_model = pd.crosstab(
    exp_top_models['model_family'],
    exp_top_models['balancing_strategy'],
    normalize='index'
) * 100
print((pct_model.round(1).astype(str) + '%').to_string())
print()

Heterogeneidade: Estratégias por modelo - Top 5 (% por linha)
balancing_strategy cost_sensitive data_augmentation ensemble_based generative hybrid   none  other oversampling threshold_moving undersampling
model_family                                                                                                                                  
cnn                         25.6%              3.7%           1.0%       1.0%   0.0%  30.0%  23.6%         4.0%             8.8%          2.4%
ensemble                     0.0%              0.0%          10.5%       0.9%   1.8%  29.8%  18.9%        31.6%             0.4%          6.1%
kernel                       4.4%              0.0%           4.9%       0.5%   1.6%  30.6%  20.8%        31.7%             0.0%          5.5%
mlp                          5.6%              0.0%           0.0%       3.1%   2.5%  29.8%  29.2%        23.6%             0.6%          5.6%
other                        5.8%              1.4%           1.0%       1.9%   

## 5. Heterogeneidade: Estratégias × Tipo de Tarefa

In [31]:
print("Heterogeneidade: Estratégias por tipo de tarefa (contagem)")
crosstab_task = pd.crosstab(
    exp['task_type'],
    exp['balancing_strategy'],
    margins=True
)
print(crosstab_task)
print()

Heterogeneidade: Estratégias por tipo de tarefa (contagem)
balancing_strategy  cost_sensitive  data_augmentation  ensemble_based  \
task_type                                                               
other                           62                  8              28   
tabular                         26                  4              32   
text                             2                  1               0   
time_series                      0                  0               0   
vision                          22                  2               1   
All                            112                 15              61   

balancing_strategy  generative  hybrid  none  other  oversampling  \
task_type                                                           
other                        0       1    80     98            54   
tabular                     13      18   252    150           243   
text                         0       0     7      7             2   
time_series

In [32]:
print("Heterogeneidade: Estratégias por task type (% por linha)")
pct_task = pd.crosstab(
    exp['task_type'],
    exp['balancing_strategy'],
    normalize='index'
) * 100
print((pct_task.round(1).astype(str) + '%').to_string())
print()

Heterogeneidade: Estratégias por task type (% por linha)
balancing_strategy cost_sensitive data_augmentation ensemble_based generative hybrid   none  other oversampling threshold_moving two_stage undersampling
task_type                                                                                                                                               
other                       17.1%              2.2%           7.7%       0.0%   0.3%  22.0%  27.0%        14.9%             4.7%      0.0%          4.1%
tabular                      3.2%              0.5%           4.0%       1.6%   2.2%  31.3%  18.7%        30.2%             1.0%      0.2%          7.0%
text                        10.0%              5.0%           0.0%       0.0%   0.0%  35.0%  35.0%        10.0%             0.0%      0.0%          5.0%
time_series                  0.0%              0.0%           0.0%       0.0%   0.0%   0.0%  50.0%        50.0%             0.0%      0.0%          0.0%
vision                   

## 6. Cobertura de Dados Estruturados e Métricas

In [33]:
print("Cobertura de dados estruturados:")
print(f"  dataset_size:            {exp['dataset_size'].notna().mean():.1%}")
print(f"  dataset_num_classes:     {exp['dataset_num_classes'].notna().mean():.1%}")
print(f"  dataset_imbalance_ratio: {exp['dataset_imbalance_ratio'].notna().mean():.1%}")
print(f"  dataset_is_multilabel:   {exp['dataset_is_multilabel'].notna().mean():.1%}")
print()

Cobertura de dados estruturados:
  dataset_size:            67.1%
  dataset_num_classes:     90.2%
  dataset_imbalance_ratio: 78.5%
  dataset_is_multilabel:   100.0%



In [34]:
print("Cobertura de métricas primárias:")
for metric in ['metric_f1_macro', 'metric_f1_weighted', 'metric_balanced_acc', 
               'metric_accuracy', 'metric_aucroc']:
    pct = exp[metric].notna().mean()
    print(f"  {metric:<25} {pct:.1%}")
print()

Cobertura de métricas primárias:
  metric_f1_macro           5.5%
  metric_f1_weighted        0.6%
  metric_balanced_acc       5.2%
  metric_accuracy           44.0%
  metric_aucroc             30.4%



In [35]:
print(f"Experimentos com baseline no grupo: {exp['has_baseline_in_group'].mean():.1%}")

Experimentos com baseline no grupo: 66.1%
